In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install langchain==0.3.27 langchain-docling docling faiss-cpu sentence-transformers
!pip install langchain-text-splitters langchain-huggingface langchain-classic
!pip install langchain-community==0.3.27 langchain-core==0.3.75 gradio nest_asyncio
!pip install -U transformers accelerate
!pip install ragas==0.2.15 datasets

!pip install rank_bm25

!pip install -U bitsandbytes 



In [ ]:

# ============================================================
# Imports  
# ============================================================
import os
import torch
import gradio as gr

os.environ.setdefault("USER_AGENT", "ml-rag-chatbot/1.0")

from langchain_docling import DoclingLoader
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_classic.memory import ConversationBufferMemory
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from sentence_transformers import CrossEncoder

import warnings
warnings.filterwarnings("ignore")

# ============================================================
# Document ingestion
# ============================================================
print("Loading documents...")

loader_pdf = DoclingLoader(
    file_path=[
        "https://arxiv.org/pdf/1810.04805.pdf",    # BERT
        "https://arxiv.org/pdf/1402.1128.pdf",  # LSTM-related
        "https://arxiv.org/pdf/2106.09685.pdf" ,  # Lora
        "https://arxiv.org/pdf/2305.14314.pdf",  # QLora
        "https://arxiv.org/pdf/2305.18438.pdf"  # RLHF

    ]
)

loader_web = WebBaseLoader([
    "https://en.wikipedia.org/wiki/Machine_learning",
    "https://en.wikipedia.org/wiki/Long_short-term_memory",
    "https://en.wikipedia.org/wiki/Gradient_descent",
    "https://en.wikipedia.org/wiki/Backpropagation",
    "https://en.wikipedia.org/wiki/Overfitting",
    "https://en.wikipedia.org/wiki/Dropout_(neural_networks)",
    "https://en.wikipedia.org/wiki/Supervised_learning",
    "https://en.wikipedia.org/wiki/Transformer_(deep_learning_architecture)"
])

docs = loader_pdf.load()
docs.extend(loader_web.load())
print(f"Total documents loaded: {len(docs)}")

# ============================================================
# Text cleaning
# ============================================================
import re

print("Preprocessing and cleaning document text content...")

for doc in docs:
    text = doc.page_content
    text = re.sub(r'\[\d+\]', '', text)
    text = re.sub(r'\s*\[\s*edit\s*\]', '', text, flags=re.IGNORECASE)
    text = re.sub(r'[\n\r\t]+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\s+\.\s+', '. ', text)
    text = re.sub(r'\s+,\s+', ', ', text)
    doc.page_content = text.strip()

print("Cleaning complete.")

# ============================================================
# Document Splitting & Vector Embeddings
# ============================================================
print("Splitting documents into chunks...")

tokenizer_split = AutoTokenizer.from_pretrained("BAAI/bge-small-en-v1.5")

text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer_split,
    chunk_size=512,
    chunk_overlap=80
)

chunks = text_splitter.split_documents(docs)
print(f"Total chunks created: {len(chunks)}")

print("Creating semantic embedding indices via BGE-Small...")
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

# ============================================================
# Advanced Hybrid Ensemble Retriever (BM25 + FAISS)
# ============================================================
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

print("Building Hybrid Search Vector Stores...")

# 1. Initialize our Sparse Keyword Index (BM25)
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 4  # Pull the top 4 exact keyword matching paragraphs

# 2. Initialize our Dense Semantic Index (FAISS) using Max Marginal Relevance
faiss_db = FAISS.from_documents(chunks, embeddings)
faiss_mmr_retriever = faiss_db.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 20,
        "lambda_mult": 0.7  # Strongly prioritize immediate query relevance over variety
    }
)

# 3. Ensemble both pipelines to merge keyword precision with deep semantics
# Weight distribution favors dense mapping while using BM25 to recover pure term lookups
retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_mmr_retriever],
    weights=[0.4, 0.6]
)

print("Hybrid Ensemble Retriever initialized successfully.")

# ============================================================
# CrossEncoder Re-ranking
# ============================================================

print("Loading CrossEncoder reranker...")

cross_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    device="cuda" if torch.cuda.is_available() else "cpu"
)


class CrossEncoderRetriever:
    """
    Wraps an existing retriever and reranks the retrieved
    documents using a CrossEncoder.

    NOTE: this is a plain Python object, not a LangChain Runnable, so it
    does NOT support the `|` pipe operator used in LCEL chains. Anywhere
    this needs to be composed into a chain, wrap it as
    RunnableLambda(retriever.invoke) instead of writing `retriever | ...`.
    """

    def __init__(self, base_retriever, cross_encoder, top_k=5):
        self.base_retriever = base_retriever
        self.cross_encoder = cross_encoder
        self.top_k = top_k

    def invoke(self, query):

        docs = self.base_retriever.invoke(query)

        if len(docs) == 0:
            return docs

        sentence_pairs = [
            (query, doc.page_content)
            for doc in docs
        ]

        scores = self.cross_encoder.predict(sentence_pairs)

        ranked = sorted(
            zip(scores, docs),
            key=lambda x: x[0],
            reverse=True
        )

        return [
            doc
            for score, doc in ranked[:self.top_k]
        ]


retriever = CrossEncoderRetriever(
    retriever,
    cross_encoder,
    top_k=5
)

print("CrossEncoder reranker initialized.")


# ============================================================
# LLM loading
# ============================================================
print("Loading Quantized LLM...")

model_id = "ibm-granite/granite-3.0-8b-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# Configure 4-bit NF4 quantization.
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto" if torch.cuda.is_available() else None,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    trust_remote_code=True
)

text_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_length=None,
    max_new_tokens=256,
    temperature=0.3,
    do_sample=True,
    repetition_penalty=1.2,
    pad_token_id=tokenizer.eos_token_id,
    return_full_text=False
)

llm = HuggingFacePipeline(pipeline=text_pipeline)

# ============================================================
# Memory & chain utilities
# ============================================================
def make_memory():
    return ConversationBufferMemory(
        memory_key="chat_history",
        return_messages=False
    )

def format_docs(docs):

    if not docs:
        return ""

    formatted = []

    for i, doc in enumerate(docs, 1):

        source = "Unknown"

        if hasattr(doc, "metadata"):
            source = doc.metadata.get("source", doc.metadata.get("file_path", "Unknown"))

        # the original concatenated "SOURCE [i]{source}CONTENT{text}"
        # with no separators, gluing the URL directly onto the chunk text
        # (e.g. "SOURCE [1]https://...CONTENTBERT is..."). Added a colon
        # and newlines so the LLM can actually tell source from content.
        formatted.append(f"SOURCE [{i}]: {source}\nCONTENT:\n{doc.page_content}")

    return "\n\n".join(formatted)

def clean_output(output: str) -> str:
    # added Granite-specific end tokens
    output = (output
              .replace("<|end_of_text|>", "")
              .replace("<|endoftext|>", "")
              .replace("<|assistant|>", "")
              .strip())
    stop_markers = ["Context:", "Question:", "Chat History:", "You are a", "<|system|>", "<|user|>"]
    for marker in stop_markers:
        if marker in output:
            output = output.split(marker)[0]
    return output.strip()


# ============================================================
# RAG chain
# Format prompts using Granite's native chat template to ensure
# proper instruction-following during generation.
# ============================================================
def format_granite_prompt(inputs: dict) -> str:
    """
    Converts chain inputs into a single string using Granite's
    native chat template. This is what makes the model actually
    follow instructions instead of returning empty output.
    """
    chat_history = inputs.get("chat_history", "")
    context      = inputs.get("context", "")
    question     = inputs.get("question", "")

    messages = [
        {
            "role": "system",
            "content": (
                "You are a Machine Learning question-answering assistant. "
                "Answer strictly from the provided context in 3-4 sentences max. "
                "Do not repeat the question. Do not ask follow-up questions. "
                "If the question is unrelated to ML or DL, say so politely."
            )
        },
        {
            "role": "user",
            "content": (
                f"Chat History:\n{chat_history}\n\n"
                f"Context:\n{context}\n\n"
                f"Question:\n{question}"
            )
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True   # adds <|assistant|> so model knows to reply
    )

def build_rag_chain(memory: ConversationBufferMemory):
    chain = (
        {
            # Retrieve relevant documents, format them as context,
            # and include conversation history in the prompt.
            "context":      RunnableLambda(lambda x: format_docs(retriever.invoke(x))),
            "question":     RunnablePassthrough(),
            "chat_history": RunnableLambda(
                lambda _: memory.load_memory_variables({})["chat_history"]
            ),
        }
        # Format the prompt using Granite's chat template.
        | RunnableLambda(format_granite_prompt)  
        | llm         # Generate the response.
        | StrOutputParser()             # Parse and clean the generated output.
        | RunnableLambda(clean_output)
    )
    return chain

# ============================================================
# Chatbot function
# ============================================================
def chatbot(message: str, history: list, memory: ConversationBufferMemory):
    history.append({"role": "user", "content": message})

    try:
        # Retrieve documents separately so we can show sources
        retrieved_docs = retriever.invoke(message)

        # Build context
        context = format_docs(retrieved_docs)

        prompt = format_granite_prompt({
            "chat_history": memory.load_memory_variables({})["chat_history"],
            "context": context,
            "question": message
        })

        answer = clean_output(llm.invoke(prompt))

        if not answer.strip():
            answer = "The model returned an empty response."

        # -------- Add Sources --------
        sources = []

        for doc in retrieved_docs:
            src = doc.metadata.get("source", "Unknown")
            page = doc.metadata.get("page", "")

            if page != "":
                sources.append(f"{src} (Page {page})")
            else:
                sources.append(src)

        if sources:
            answer += "\n\n📚 Sources:\n"
            answer += "\n".join(f"- {s}" for s in sorted(set(sources)))

    except Exception as e:
        answer = f"An error occurred: {e}"

    memory.save_context(
        {"input": message},
        {"output": answer}
    )

    history.append(
        {
            "role": "assistant",
            "content": answer
        }
    )

    return history, memory


# ============================================================
# RAGAS Evaluation
# Uses the official RAGAS framework to assess the quality of the
# RAG pipeline. A separate instruction-tuned judge model is used
# to evaluate generated responses for:
# - Faithfulness (grounding in retrieved context)
# - Response Relevancy (relevance to the user query)
# ============================================================
print("STARTED_EVALUATION_______")
import warnings
import logging
import nest_asyncio

nest_asyncio.apply()
warnings.filterwarnings("ignore")

# Configure RAGAS logging to display parsing warnings while
# suppressing unnecessary output.
logging.getLogger("ragas").setLevel(logging.WARNING)

# Reduce verbose Transformers logging during evaluation.
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

from datasets import Dataset

from ragas import evaluate
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
)

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig


print("=" * 60)
print("Running Official RAGAS Evaluation")
print("=" * 60)

print("=" * 60)
print("Loading Judge Model")
print("=" * 60)
# ============================================================
# Load Judge LLM for RAGAS Evaluation
# This model is used only to evaluate generated responses and
# compute RAGAS metrics. It is not used for answer generation.
# ============================================================
judge_model_id = "Qwen/Qwen2.5-1.5B-Instruct"

judge_tokenizer = AutoTokenizer.from_pretrained(
    judge_model_id,
    trust_remote_code=True
)

judge_model = AutoModelForCausalLM.from_pretrained(
    judge_model_id,
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)


# ------------------------------------------------------------
# Dedicated deterministic generation pipeline for the RAGAS judge model.
# Greedy decoding improves structured-output reliability.
# ------------------------------------------------------------
judge_pipeline = pipeline(
    "text-generation",
    model=judge_model,
    tokenizer=judge_tokenizer,
    max_new_tokens=512,
    do_sample=False,
    temperature=0.0,
    repetition_penalty=1.0,
    return_full_text=False,
    pad_token_id=judge_tokenizer.eos_token_id,
)

judge_llm = HuggingFacePipeline(
    pipeline=judge_pipeline
)


# ------------------------------------------------------------
# Wrap LangChain components for RAGAS
# ------------------------------------------------------------

ragas_llm = LangchainLLMWrapper(judge_llm)
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)

# newer RAGAS no longer propagates llm= / embeddings= from evaluate()
# down into metric instances — you must set them directly on each metric.
faithfulness_metric = Faithfulness()
faithfulness_metric.llm = ragas_llm

response_relevancy_metric = ResponseRelevancy()
response_relevancy_metric.llm = ragas_llm
response_relevancy_metric.embeddings = ragas_embeddings

# Run evaluation sequentially for local inference.
run_config = RunConfig(max_workers=1, timeout=300)


# ------------------------------------------------------------
# Evaluation Questions
# ------------------------------------------------------------

EVAL_QUESTIONS = [
    "What is supervised learning?",
    "How does BERT work?",
    "What is an LSTM?",
    "What is Gradient Descent?",
    "What is Backpropagation?",
    "What is the main idea behind Low-Rank Adaptation (LoRA)?",
    "What is QLoRA?",
    "What is self-attention in the Transformer architecture?",
    "Explain scaled dot-product attention.",
    "What is the purpose of the forget gate in an LSTM?"
]


# ------------------------------------------------------------
# Build Evaluation Dataset
# ------------------------------------------------------------

records = []

# Configure the evaluation dataset by limiting the number of
# retrieved contexts and the amount of text included for each
# question to keep evaluation efficient.
EVAL_MAX_CONTEXTS = 3
EVAL_MAX_CHARS_PER_CONTEXT = 600

for question in EVAL_QUESTIONS:

    print(f"Generating answer for: {question}")

    docs = retriever.invoke(question)        # Retrieve the most relevant documents for the evaluation query.

    contexts = [
        doc.page_content[:EVAL_MAX_CHARS_PER_CONTEXT]
        for doc in docs[:EVAL_MAX_CONTEXTS]
    ]                                                 # Select a subset of retrieved contexts for RAGAS evaluation.

    chain = build_rag_chain(make_memory())             # Build a fresh RAG chain for each evaluation sample.

    try:
        answer = chain.invoke(question)
    except Exception as e:
        print(e)
        answer = ""

    records.append(
        {
            "user_input": question,
            "response": answer,
            "retrieved_contexts": contexts
        }
    )                     # Store the evaluation sample in the required RAGAS format.

# Create a Hugging Face Dataset for RAGAS evaluation.
dataset = Dataset.from_list(records)


# ------------------------------------------------------------
# Run Official RAGAS
# ------------------------------------------------------------

results = evaluate(
    dataset=dataset,
    metrics=[
        faithfulness_metric,
        response_relevancy_metric,
    ],
    run_config=run_config,
    raise_exceptions=False,
)


# ------------------------------------------------------------
# Display Results
# ------------------------------------------------------------

df = results.to_pandas()

print("\n")
print("=" * 60)
print("Per Question Scores")
print("=" * 60)

print(df)

print("\n")
print("=" * 60)
print("Average Scores")
print("=" * 60)

print(df.mean(numeric_only=True))

print("\n")
print("=" * 60)
print("Evaluation Complete")
print("=" * 60)



# ============================================================
# Gradio UI
# ============================================================
with gr.Blocks(title="ML RAG Chatbot") as demo:
    gr.Markdown("## Machine Learning RAG Chatbot")
    gr.Markdown("Ask questions about Machine Learning and Deep Learning.")

    chatbot_ui = gr.Chatbot(height=450, type="messages", allow_tags=False)  

    with gr.Row():
        msg_box  = gr.Textbox(placeholder="Ask a machine learning question...", show_label=False, scale=8)
        send_btn = gr.Button("Send", variant="primary", scale=1)   # explicit send button

    clear_btn      = gr.Button("Clear conversation")
    session_memory = gr.State(make_memory())

    def user_submit(message, history, memory):
        if not message.strip():
            return history, memory, gr.update(value="")
        updated_history, updated_memory = chatbot(message, history, memory)
        return updated_history, updated_memory, gr.update(value="")

    # Enter key inside the textbox
    msg_box.submit(
        fn=user_submit,
        inputs=[msg_box, chatbot_ui, session_memory],
        outputs=[chatbot_ui, session_memory, msg_box],
    )

    # clicking Send does exactly the same thing as pressing Enter
    send_btn.click(
        fn=user_submit,
        inputs=[msg_box, chatbot_ui, session_memory],
        outputs=[chatbot_ui, session_memory, msg_box],
    )

    # Reset the chat history and conversation memory.
    clear_btn.click(
        fn=lambda: ([], make_memory()),
        outputs=[chatbot_ui, session_memory]
    )

    gr.Examples(
        examples=[
            "What is machine learning?",
            "What are the types of machine learning?",
            "What is supervised learning?",
            "What is overfitting?",
            "How does BERT work?",
            "What is an LSTM?",
        ],
        inputs=[msg_box],
        fn=user_submit,                                 # wire to user_submit so click submits
        outputs=[chatbot_ui, session_memory, msg_box],
        cache_examples=False,                           # don't precompute on Kaggle
    )

demo.queue()

demo.launch(
    share=True,
    debug=True,
    show_error=True,
    prevent_thread_lock=True
)


Loading documents...


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

Total documents loaded: 576
Preprocessing and cleaning document text content...
Cleaning complete.
Splitting documents into chunks...
Total chunks created: 850
Creating semantic embedding indices via BGE-Small...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Building Hybrid Search Vector Stores...
Hybrid Ensemble Retriever initialized successfully.
Loading CrossEncoder reranker...


...

STARTED_EVALUATION_______
Running Official RAGAS Evaluation
Loading Judge Model


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Generating answer for: What is supervised learning?
Generating answer for: How does BERT work?
Generating answer for: What is an LSTM?
Generating answer for: What is Gradient Descent?
Generating answer for: What is Backpropagation?
Generating answer for: What is the main idea behind Low-Rank Adaptation (LoRA)?
Generating answer for: What is QLoRA?
Generating answer for: What is self-attention in the Transformer architecture?
Generating answer for: Explain scaled dot-product attention.
Generating answer for: What is the purpose of the forget gate in an LSTM?


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]



Per Question Scores
                                          user_input  \
0                       What is supervised learning?   
1                                How does BERT work?   
2                                   What is an LSTM?   
3                          What is Gradient Descent?   
4                           What is Backpropagation?   
5  What is the main idea behind Low-Rank Adaptati...   
6                                     What is QLoRA?   
7  What is self-attention in the Transformer arch...   
8              Explain scaled dot-product attention.   
9  What is the purpose of the forget gate in an L...   

                                  retrieved_contexts  \
0  [output. The term "supervised" refers to the r...   
1  [3 BERT Input/Output Representations To make B...   
2  [NeurIPS ICML ICLR IJCAI ML JMLR Related artic...   
3  [of the function at the current point, because...   
4  [in computer vision and image processing Outli...   
5  [1 INTRODUCTION We tak